In [3]:
import pandas as pd

steam = pd.read_csv('steam_games_dataset.csv')
rawg = pd.read_csv('rawg_games.csv')
gog = pd.read_csv('gog_data.csv')

print("STEAM:")
print(steam.head())
print(steam.info())

print("\nRAWG:")
print(rawg.head())
print(rawg.info())

print("\nGOG:")
print(gog.head())
print(gog.info())

STEAM:
    app_id                   name release_date  release_year  price  \
0      730       Counter-Strike 2   2012-08-21        2012.0   0.00   
1  3764200  Resident Evil Requiem   2026-02-27        2026.0  69.99   
2  3065800               Marathon   2026-03-05        2026.0  39.99   
3   553850          HELLDIVERS™ 2   2024-02-08        2024.0  39.99   
4  1808500            ARC Raiders   2025-10-30        2025.0  39.99   

   positive_pct  total_reviews             review_label  \
0          86.0      2502519.0            Very Positive   
1          96.0        41952.0  Overwhelmingly Positive   
2          90.0        21313.0            Very Positive   
3          82.0       613768.0            Very Positive   
4          86.0       178220.0            Very Positive   

                                   short_description  \
0  For over two decades, Counter-Strike has offer...   
1  Requiem for the dead. Nightmare for the living...   
2  Scavenge the lost colony of Tau Ceti IV 

In [4]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

tqdm.pandas()

steam_df = steam.copy()
rawg_df = rawg.copy()
gog_df = gog.copy()

def normalize_name(name):
    if pd.isna(name):
        return np.nan

    name = str(name).lower().strip()
    name = re.sub(r'[™®©]', '', name)
    name = re.sub(r'[-–—_:]', ' ', name)
    name = re.sub(r'[^\w\s]', '', name)

    noise_patterns = [
        r'\bdeluxe edition\b',
        r'\bcomplete edition\b',
        r'\bultimate edition\b',
        r'\bgame of the year edition\b',
        r'\bgoty edition\b',
        r'\bsoundtrack\b',
        r'\bdemo\b',
        r'\bbeta\b',
        r'\btrial\b'
    ]

    for pattern in noise_patterns:
        name = re.sub(pattern, '', name)

    name = re.sub(r'\s+', ' ', name).strip()
    return name


def clean_text(text):
    if pd.isna(text):
        return np.nan
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)     # html
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_price(price):
    if pd.isna(price):
        return np.nan
    price = str(price)
    match = re.search(r'[\d]+(?:[.,]\d+)?', price)
    if match:
        return float(match.group().replace(',', '.'))
    return np.nan


def mark_special_edition(title):
    if pd.isna(title):
        return 0
    title = str(title).lower()
    patterns = [
        'soundtrack', 'demo', 'beta', 'dlc', 'expansion',
        'bundle', 'pack'
    ]
    return int(any(p in title for p in patterns))

steam_df = steam_df.rename(columns={
    'name': 'steam_name',
    'price': 'price_steam',
    'positive_pct': 'steam_positive_pct',
    'total_reviews': 'steam_total_reviews',
    'review_label': 'steam_review_label',
    'short_description': 'steam_short_description',
    'genres': 'steam_genres',
    'tags': 'steam_tags',
    'developer': 'steam_developer'
})

rawg_df = rawg_df.rename(columns={
    'name': 'rawg_name',
    'released': 'rawg_released',
    'ratings_count': 'rawg_ratings_count',
    'genres': 'rawg_genres',
    'description': 'rawg_description',
    'playtime': 'avg_playtime_hours'
})

gog_df = gog_df.rename(columns={
    'title': 'gog_name',
    'price': 'price_gog',
    'rating': 'gog_rating',
    'reviews_count': 'gog_reviews_count',
    'genre': 'gog_genres',
    'developer': 'gog_developer',
    'description': 'gog_description',
    'year': 'gog_year'
})

rawg_df['rawg_description'] = rawg_df['rawg_description'].progress_apply(clean_text)
steam_df['steam_short_description'] = steam_df['steam_short_description'].progress_apply(clean_text)
gog_df['gog_description'] = gog_df['gog_description'].progress_apply(clean_text)

gog_df['price_gog'] = gog_df['price_gog'].progress_apply(clean_price)

gog_df['is_special_edition'] = gog_df['gog_name'].progress_apply(mark_special_edition)

steam_df['normalized_name'] = steam_df['steam_name'].progress_apply(normalize_name)
rawg_df['normalized_name'] = rawg_df['rawg_name'].progress_apply(normalize_name)
gog_df['normalized_name'] = gog_df['gog_name'].progress_apply(normalize_name)

steam_df['release_date'] = pd.to_datetime(steam_df['release_date'], errors='coerce')
rawg_df['rawg_released'] = pd.to_datetime(rawg_df['rawg_released'], errors='coerce')

steam_df = steam_df.drop_duplicates(subset=['app_id'])
rawg_df = rawg_df.drop_duplicates(subset=['rawg_id'])
gog_df = gog_df.drop_duplicates(subset=['gog_name', 'url'])

gog_main = gog_df[gog_df['is_special_edition'] == 0].copy()

print("steam_df:", steam_df.shape)
print("rawg_df:", rawg_df.shape)
print("gog_df:", gog_df.shape)
print("gog_main:", gog_main.shape)

print("\nПримеры normalized_name:")
print(steam_df[['steam_name', 'normalized_name']].head())
print(rawg_df[['rawg_name', 'normalized_name']].head())
print(gog_main[['gog_name', 'normalized_name']].head())

100%|██████████| 9268/9268 [00:00<00:00, 52541.97it/s]


steam_df: (11250, 13)
rawg_df: (360, 17)
gog_df: (9268, 12)
gog_main: (7110, 12)

Примеры normalized_name:
              steam_name        normalized_name
0       Counter-Strike 2       counter strike 2
1  Resident Evil Requiem  resident evil requiem
2               Marathon               marathon
3          HELLDIVERS™ 2           helldivers 2
4            ARC Raiders            arc raiders
                                   rawg_name  \
0                       The Elder Scrolls VI   
1             No Case Should Remain Unsolved   
2                                   Gimmick!   
3    Super Robot Taisen: Original Generation   
4  The Witcher 3: Wild Hunt – Blood and Wine   

                          normalized_name  
0                    the elder scrolls vi  
1          no case should remain unsolved  
2                                 gimmick  
3  super robot taisen original generation  
4  the witcher 3 wild hunt blood and wine  
                                      gog_name      

In [5]:
def normalize_name(name):
    if pd.isna(name):
        return np.nan

    name = str(name).lower().strip()
    name = re.sub(r'[™®©]', '', name)
    name = re.sub(r'[-–—_:]', ' ', name)
    name = re.sub(r'[^\w\s]', '', name)

    noise_patterns = [
        r'\bdeluxe edition\b',
        r'\bcomplete edition\b',
        r'\bultimate edition\b',
        r'\bgame of the year edition\b',
        r'\bgoty edition\b',
        r'\bgold edition\b',
        r'\bplatinum edition\b',
        r'\bdefinitive edition\b',
        r'\bcollectors edition\b',
        r'\bdirectors cut\b',
        r'\bsoundtrack\b',
        r'\bdemo\b',
        r'\bbeta\b',
        r'\btrial\b'
    ]

    for pattern in noise_patterns:
        name = re.sub(pattern, '', name)

    name = re.sub(r'\s+', ' ', name).strip()
    return name

In [6]:
steam_df['normalized_name'] = steam_df['steam_name'].progress_apply(normalize_name)
rawg_df['normalized_name'] = rawg_df['rawg_name'].progress_apply(normalize_name)
gog_df['normalized_name'] = gog_df['gog_name'].progress_apply(normalize_name)
gog_main['normalized_name'] = gog_main['gog_name'].progress_apply(normalize_name)

100%|██████████| 7110/7110 [00:00<00:00, 42264.45it/s]


In [7]:
print("Steam duplicate normalized_name:", steam_df['normalized_name'].duplicated().sum())
print("RAWG duplicate normalized_name:", rawg_df['normalized_name'].duplicated().sum())
print("GOG duplicate normalized_name:", gog_main['normalized_name'].duplicated().sum())

Steam duplicate normalized_name: 54
RAWG duplicate normalized_name: 1
GOG duplicate normalized_name: 236


In [8]:
steam_dups = steam_df[steam_df['normalized_name'].duplicated(keep=False)].sort_values('normalized_name')
rawg_dups = rawg_df[rawg_df['normalized_name'].duplicated(keep=False)].sort_values('normalized_name')
gog_dups = gog_main[gog_main['normalized_name'].duplicated(keep=False)].sort_values('normalized_name')

print("STEAM duplicates sample:")
print(steam_dups[['steam_name', 'normalized_name']].head(20))

print("\nRAWG duplicates sample:")
print(rawg_dups[['rawg_name', 'normalized_name']].head(20))

print("\nGOG duplicates sample:")
print(gog_dups[['gog_name', 'normalized_name']].head(20))

STEAM duplicates sample:
                                           steam_name  \
1514                                             20XX   
4954                                  20XX Soundtrack   
6145                       A Hat in Time - Soundtrack   
1022                                    A Hat in Time   
178             Age of Empires II: Definitive Edition   
1543            Age of Empires II: Definitive Edition   
10035                    Battle Brothers - Soundtrack   
578                                   Battle Brothers   
1399                Bloodstained: Ritual of the Night   
7066   Bloodstained: Ritual of the Night - Soundtrack   
1131                       Call to Arms: Panzer Elite   
2356      Call to Arms: Panzer Elite - Deluxe Edition   
627                                           Celeste   
4076                               Celeste Soundtrack   
181                                   DARK SOULS™ III   
3199                    DARK SOULS III Deluxe Edition   
7678  

In [9]:
steam_keys = set(steam_df['normalized_name'].dropna())
rawg_keys = set(rawg_df['normalized_name'].dropna())
gog_keys = set(gog_main['normalized_name'].dropna())

steam_rawg_overlap = steam_keys & rawg_keys
steam_gog_overlap = steam_keys & gog_keys
rawg_gog_overlap = rawg_keys & gog_keys

print("Steam unique keys:", len(steam_keys))
print("RAWG unique keys:", len(rawg_keys))
print("GOG unique keys:", len(gog_keys))

print("\nSteam ∩ RAWG:", len(steam_rawg_overlap))
print("Steam ∩ GOG:", len(steam_gog_overlap))
print("RAWG ∩ GOG:", len(rawg_gog_overlap))

Steam unique keys: 11196
RAWG unique keys: 359
GOG unique keys: 6874

Steam ∩ RAWG: 80
Steam ∩ GOG: 1966
RAWG ∩ GOG: 29


In [10]:
print("Steam ∩ RAWG sample:", list(sorted(steam_rawg_overlap))[:20])
print("Steam ∩ GOG sample:", list(sorted(steam_gog_overlap))[:20])

Steam ∩ RAWG sample: ['beyond good evil 20th anniversary edition', 'bring you home', 'clair obscur expedition 33', 'command conquer generals', 'cuphead the delicious last course', 'cyberpunk 2077 phantom liberty', 'dark souls iii ashes of ariandel', 'dark souls iii the ringed city', 'dark souls remastered', 'disciples ii galleans return', 'disciples ii rise of the elves', 'dispatch', 'divinity original sin 2', 'dragon age origins', 'elden ring shadow of the erdtree', 'esoteric ebb', 'fable the lost chapters', 'fallen hero rebirth', 'fallout new vegas', 'fear effect 2 retro helix']
Steam ∩ GOG sample: ['7 billion humans', '8 bit adventures 1 the forgotten journey remastered edition', '8 bit adventures 2', '80 days', '9 kings', '9th dawn iii', 'a bird story', 'a building full of cats', 'a building full of cats 2', 'a castle full of cats', 'a hat in time', 'a hat in time seal the deal', 'a legionarys life', 'a park full of cats', 'a plague tale innocence', 'a plague tale requiem', 'a shel

In [11]:
rawg_unique = (
    rawg_df
    .sort_values(['normalized_name', 'rawg_ratings_count', 'rawg_rating'], ascending=[True, False, False])
    .drop_duplicates(subset=['normalized_name'], keep='first')
    .copy()
)

print("rawg_df:", rawg_df.shape)
print("rawg_unique:", rawg_unique.shape)

rawg_df: (360, 17)
rawg_unique: (359, 17)


In [12]:
gog_unique = (
    gog_main
    .sort_values(['normalized_name', 'gog_reviews_count', 'gog_rating'], ascending=[True, False, False])
    .drop_duplicates(subset=['normalized_name'], keep='first')
    .copy()
)

print("gog_main:", gog_main.shape)
print("gog_unique:", gog_unique.shape)

gog_main: (7110, 12)
gog_unique: (6874, 12)


In [13]:
steam_unique = (
    steam_df
    .sort_values(['normalized_name', 'steam_total_reviews', 'steam_positive_pct'], ascending=[True, False, False])
    .drop_duplicates(subset=['normalized_name'], keep='first')
    .copy()
)

print("gog_main:", gog_main.shape)
print("gog_unique:", gog_unique.shape)

gog_main: (7110, 12)
gog_unique: (6874, 12)


In [14]:
print("Steam duplicate normalized_name:", steam_unique['normalized_name'].duplicated().sum())
print("RAWG duplicate normalized_name:", rawg_unique['normalized_name'].duplicated().sum())
print("GOG duplicate normalized_name:", gog_unique['normalized_name'].duplicated().sum())

Steam duplicate normalized_name: 0
RAWG duplicate normalized_name: 0
GOG duplicate normalized_name: 0


In [15]:
steam_rawg = steam_unique.merge(
    rawg_unique,
    on='normalized_name',
    how='left',
    suffixes=('', '_rawgdup')
)

print("steam_unique:", steam_df.shape)
print("steam_rawg:", steam_rawg.shape)

steam_unique: (11250, 13)
steam_rawg: (11196, 29)


In [37]:
merged_df = steam_rawg.merge( gog_unique, on='normalized_name', how='left', suffixes=('', '_gogdup') )

print("steam_rawg:", steam_rawg.shape)
print("merged_df:", merged_df.shape)

steam_rawg: (11196, 30)
merged_df: (11196, 41)


In [36]:
merged_df['matched_gog'] = merged_df['gog_name'].notna().astype(int)

print("Matched with GOG:", merged_df['matched_gog'].sum())
print("Match rate with GOG:", merged_df['matched_gog'].mean())

Matched with GOG: 1966
Match rate with GOG: 0.17559842801000358


In [19]:
final_df = pd.DataFrame()

final_df['app_id'] = merged_df['app_id']
final_df['name'] = merged_df['steam_name'].combine_first(merged_df['rawg_name']).combine_first(merged_df['gog_name'])
final_df['normalized_name'] = merged_df['normalized_name']

final_df['release_date'] = merged_df['release_date'].combine_first(merged_df['rawg_released'])
final_df['release_year'] = merged_df['release_year']
final_df['release_year'] = final_df['release_year'].fillna(pd.to_datetime(final_df['release_date'], errors='coerce').dt.year)
final_df['release_year'] = final_df['release_year'].combine_first(merged_df['gog_year'])

final_df['price_steam'] = merged_df['price_steam']
final_df['price_gog'] = merged_df['price_gog']

final_df['steam_positive_pct'] = merged_df['steam_positive_pct']
final_df['steam_total_reviews'] = merged_df['steam_total_reviews']
final_df['steam_review_label'] = merged_df['steam_review_label']

final_df['rawg_rating'] = merged_df['rawg_rating']
final_df['rawg_ratings_count'] = merged_df['rawg_ratings_count']
final_df['metacritic'] = merged_df['metacritic']
final_df['avg_playtime_hours'] = merged_df['avg_playtime_hours']
final_df['platforms_count'] = merged_df['platforms_count']
final_df['achievements_count'] = merged_df['achievements_count']
final_df['esrb_rating'] = merged_df['esrb_rating']

final_df['gog_rating'] = merged_df['gog_rating']
final_df['gog_reviews_count'] = merged_df['gog_reviews_count']

final_df['description'] = (
    merged_df['rawg_description']
    .combine_first(merged_df['steam_short_description'])
    .combine_first(merged_df['gog_description'])
)

final_df['genres'] = (
    merged_df['steam_genres']
    .combine_first(merged_df['rawg_genres'])
    .combine_first(merged_df['gog_genres'])
)

final_df['tags'] = merged_df['steam_tags'].combine_first(merged_df['tags'])
final_df['developer'] = (
    merged_df['steam_developer']
    .combine_first(merged_df['gog_developer'])
)

final_df['source_steam'] = 1
final_df['source_rawg'] = merged_df['rawg_id'].notna().astype(int)
final_df['source_gog'] = merged_df['gog_name'].notna().astype(int)

print(final_df.shape)
print(final_df.head())

(11196, 26)
    app_id                     name          normalized_name release_date  \
0  3768760          007 First Light          007 first light   2026-05-27   
1   282800        100% Orange Juice         100 orange juice   2014-05-16   
2  1895900             1348 Ex Voto             1348 ex voto   2026-03-12   
3  1865060  14 Minesweeper Variants  14 minesweeper variants   2022-11-14   
4  1269370                      171                      171   2022-11-17   

   release_year  price_steam  price_gog  steam_positive_pct  \
0        2026.0        69.99        NaN                 NaN   
1        2014.0         6.99        NaN                94.0   
2        2026.0        22.49        NaN                52.0   
3        2022.0         6.99        NaN                97.0   
4        2022.0        19.99        NaN                74.0   

   steam_total_reviews       steam_review_label  ...  esrb_rating  gog_rating  \
0                  NaN                      NaN  ...          NaN

In [20]:
final_df['release_date'] = pd.to_datetime(final_df['release_date'], errors='coerce')
final_df['release_year'] = pd.to_numeric(final_df['release_year'], errors='coerce')

numeric_cols = [
    'price_steam', 'price_gog', 'steam_positive_pct', 'steam_total_reviews',
    'rawg_rating', 'rawg_ratings_count', 'metacritic', 'avg_playtime_hours',
    'platforms_count', 'achievements_count', 'gog_rating', 'gog_reviews_count'
]

for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

final_df = final_df.drop_duplicates(subset=['app_id', 'normalized_name'])

print(final_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11196 entries, 0 to 11195
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   app_id               11196 non-null  object        
 1   name                 11196 non-null  object        
 2   normalized_name      11196 non-null  object        
 3   release_date         11188 non-null  datetime64[ns]
 4   release_year         11188 non-null  float64       
 5   price_steam          11167 non-null  float64       
 6   price_gog            1966 non-null   float64       
 7   steam_positive_pct   11165 non-null  float64       
 8   steam_total_reviews  11165 non-null  float64       
 9   steam_review_label   11178 non-null  object        
 10  rawg_rating          80 non-null     float64       
 11  rawg_ratings_count   80 non-null     float64       
 12  metacritic           31 non-null     float64       
 13  avg_playtime_hours   80 non-nul

In [21]:
final_df['achievements_count'] = final_df['achievements_count'].fillna(0)
final_df['platforms_count'] = final_df['platforms_count'].fillna(0)
final_df['rawg_ratings_count'] = final_df['rawg_ratings_count'].fillna(0)
final_df['gog_reviews_count'] = final_df['gog_reviews_count'].fillna(0)

final_df['price_steam_filled'] = final_df['price_steam'].fillna(0)

In [22]:
final_df['rawg_rating_filled'] = final_df['rawg_rating'].fillna(final_df['rawg_rating'].median())
final_df['steam_total_reviews_filled'] = final_df['steam_total_reviews'].fillna(0)
final_df['price_for_score'] = final_df['price_steam'].fillna(0).clip(lower=1)

final_df['popularity_score_raw'] = (
    final_df['steam_total_reviews_filled'] * final_df['rawg_rating_filled'] / final_df['price_for_score']
)

final_df['popularity_score'] = final_df['popularity_score_raw'].rank(pct=True)

In [23]:
print("Final shape:", final_df.shape)
print("Unique app_id:", final_df['app_id'].nunique())
print("Unique normalized_name:", final_df['normalized_name'].nunique())

missing_share = (final_df.isna().mean() * 100).sort_values(ascending=False)
print(missing_share)

Final shape: (11196, 32)
Unique app_id: 11196
Unique normalized_name: 11196
metacritic                    99.723115
esrb_rating                   99.642730
avg_playtime_hours            99.285459
rawg_rating                   99.285459
gog_rating                    83.386924
price_gog                     82.440157
description                   21.114684
steam_total_reviews            0.276885
steam_positive_pct             0.276885
price_steam                    0.259021
steam_review_label             0.160772
genres                         0.142908
release_date                   0.071454
release_year                   0.071454
developer                      0.017864
normalized_name                0.000000
achievements_count             0.000000
platforms_count                0.000000
rawg_ratings_count             0.000000
name                           0.000000
app_id                         0.000000
gog_reviews_count              0.000000
tags                           0.000000
sour

In [24]:
print(final_df[['name', 'price_steam', 'rawg_rating', 'gog_rating', 'popularity_score']].head(20))

                                     name  price_steam  rawg_rating  \
0                         007 First Light        69.99          NaN   
1                       100% Orange Juice         6.99          NaN   
2                            1348 Ex Voto        22.49          NaN   
3                 14 Minesweeper Variants         6.99          NaN   
4                                     171        19.99          NaN   
5                                 1根香腸走江湖         0.00          NaN   
6                    20 Minutes Till Dawn         4.99          NaN   
7                       200% Mixed Juice!         6.99          NaN   
8                                    20XX         2.24          NaN   
9              20XX - Draco Character DLC         1.49          NaN   
10              20XX - Hawk Character DLC         1.49          NaN   
11  3030 Deathwar Redux - A Space Odyssey        14.99          NaN   
12                                   30XX        19.99          NaN   
13    

In [25]:
print(final_df.head())

    app_id                     name          normalized_name release_date  \
0  3768760          007 First Light          007 first light   2026-05-27   
1   282800        100% Orange Juice         100 orange juice   2014-05-16   
2  1895900             1348 Ex Voto             1348 ex voto   2026-03-12   
3  1865060  14 Minesweeper Variants  14 minesweeper variants   2022-11-14   
4  1269370                      171                      171   2022-11-17   

   release_year  price_steam  price_gog  steam_positive_pct  \
0        2026.0        69.99        NaN                 NaN   
1        2014.0         6.99        NaN                94.0   
2        2026.0        22.49        NaN                52.0   
3        2022.0         6.99        NaN                97.0   
4        2022.0        19.99        NaN                74.0   

   steam_total_reviews       steam_review_label  ...  \
0                  NaN                      NaN  ...   
1               8669.0            Very Positiv

In [26]:
final_df['steam_total_reviews_filled'] = final_df['steam_total_reviews'].fillna(0)
final_df['steam_positive_pct_filled'] = final_df['steam_positive_pct'].fillna(final_df['steam_positive_pct'].median())
final_df['price_for_score'] = final_df['price_steam'].fillna(0).clip(lower=1)

final_df['popularity_score_raw_v2'] = (
    np.log1p(final_df['steam_total_reviews_filled']) *
    (final_df['steam_positive_pct_filled'] / 100) /
    np.log1p(final_df['price_for_score'])
)

final_df['popularity_score_v2'] = final_df['popularity_score_raw_v2'].rank(pct=True)

In [27]:
(final_df['genres'].astype(str).str.strip() == '').mean()
(final_df['tags'].astype(str).str.strip() == '').mean()
(final_df['description'].astype(str).str.strip() == '').mean()
(final_df['developer'].astype(str).str.strip() == '').mean()

np.float64(0.0)

In [28]:
final_df['release_year'] = pd.to_numeric(final_df['release_year'], errors='coerce')
final_df['release_year'] = final_df['release_year'].astype('Int64')

In [29]:
final_df['platforms_count_filled'] = final_df['platforms_count'].where(final_df['source_rawg'] == 1, np.nan)
final_df['achievements_count_filled'] = final_df['achievements_count'].where(final_df['source_rawg'] == 1, np.nan)

In [30]:
import numpy as np

text_cols = ['description', 'genres', 'tags', 'developer', 'steam_review_label']
for col in text_cols:
    final_df[col] = final_df[col].replace(r'^\s*$', np.nan, regex=True)

final_df['release_date'] = pd.to_datetime(final_df['release_date'], errors='coerce')
final_df['release_year'] = pd.to_numeric(final_df['release_year'], errors='coerce').astype('Int64')

numeric_cols = [
    'price_steam', 'price_gog', 'steam_positive_pct', 'steam_total_reviews',
    'rawg_rating', 'rawg_ratings_count', 'metacritic', 'avg_playtime_hours',
    'platforms_count', 'achievements_count', 'gog_rating', 'gog_reviews_count'
]

for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

final_df['steam_total_reviews_filled'] = final_df['steam_total_reviews'].fillna(0)
final_df['steam_positive_pct_filled'] = final_df['steam_positive_pct'].fillna(final_df['steam_positive_pct'].median())
final_df['price_for_score'] = final_df['price_steam'].fillna(0).clip(lower=1)

final_df['popularity_score_raw_v2'] = (
    np.log1p(final_df['steam_total_reviews_filled']) *
    (final_df['steam_positive_pct_filled'] / 100) /
    np.log1p(final_df['price_for_score'])
)

final_df['popularity_score_v2'] = final_df['popularity_score_raw_v2'].rank(pct=True)

final_df['has_description'] = final_df['description'].notna().astype(int)
final_df['is_free'] = final_df['price_steam'].fillna(0).eq(0).astype(int)
final_df['has_rawg'] = final_df['source_rawg']
final_df['has_gog'] = final_df['source_gog']
final_df['description_length'] = final_df['description'].fillna('').str.split().str.len()

print(final_df.info())
print("\nMissing share after final cleanup:")
print((final_df.isna().mean() * 100).sort_values(ascending=False).head(20))

print("\nPopularity score v2 summary:")
print(final_df['popularity_score_v2'].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11196 entries, 0 to 11195
Data columns (total 42 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   app_id                      11196 non-null  object        
 1   name                        11196 non-null  object        
 2   normalized_name             11196 non-null  object        
 3   release_date                11188 non-null  datetime64[ns]
 4   release_year                11188 non-null  Int64         
 5   price_steam                 11167 non-null  float64       
 6   price_gog                   1966 non-null   float64       
 7   steam_positive_pct          11165 non-null  float64       
 8   steam_total_reviews         11165 non-null  float64       
 9   steam_review_label          11178 non-null  object        
 10  rawg_rating                 80 non-null     float64       
 11  rawg_ratings_count          11196 non-null  float64   

In [31]:
final_df.to_csv('final_dataset_full.csv', index=False)

In [32]:
eda_df = final_df[[
    'app_id', 'name', 'release_date', 'release_year',
    'price_steam', 'steam_positive_pct', 'steam_total_reviews',
    'steam_review_label', 'rawg_rating', 'metacritic',
    'avg_playtime_hours', 'platforms_count', 'achievements_count',
    'esrb_rating', 'gog_rating', 'gog_reviews_count',
    'description', 'description_length', 'genres', 'tags', 'developer',
    'is_free', 'has_description', 'has_rawg', 'has_gog',
    'popularity_score_v2'
]].copy()

eda_df.to_csv('final_dataset_for_EDA.csv', index=False)
print(eda_df.head)

<bound method NDFrame.head of         app_id                               name release_date  release_year  \
0      3768760                    007 First Light   2026-05-27          2026   
1       282800                  100% Orange Juice   2014-05-16          2014   
2      1895900                       1348 Ex Voto   2026-03-12          2026   
3      1865060            14 Minesweeper Variants   2022-11-14          2022   
4      1269370                                171   2022-11-17          2022   
...        ...                                ...          ...           ...   
11191  3932140                     魔哆魔哆 Modo Modo   2026-02-17          2026   
11192  2458530                              魔女的夜宴   2023-07-21          2023   
11193  1306750  黑色花与红山羊 / Black Datura & Red Goat   2020-08-21          2020   
11194  3202030                              龙胤立志传   2026-03-13          2026   
11195  4383960                       龙胤立志传 - 支持者包   2026-03-13          2026   

       pr

In [33]:
import numpy as np

EUR_TO_USD = 1.1555

final_df['price_steam_usd'] = final_df['price_steam'] * EUR_TO_USD
final_df['price_gog_usd'] = final_df['price_gog']
final_df['price_usd'] = final_df['price_steam_usd'].combine_first(final_df['price_gog_usd'])

final_df['steam_total_reviews_filled'] = final_df['steam_total_reviews'].fillna(0)
final_df['steam_positive_pct_filled'] = final_df['steam_positive_pct'].fillna(
    final_df['steam_positive_pct'].median()
)
final_df['price_usd_for_score'] = final_df['price_usd'].fillna(0).clip(lower=1)

final_df['popularity_score_raw_v2'] = (
    np.log1p(final_df['steam_total_reviews_filled']) *
    (final_df['steam_positive_pct_filled'] / 100) /
    np.log1p(final_df['price_usd_for_score'])
)

final_df['popularity_score_v2'] = final_df['popularity_score_raw_v2'].rank(pct=True)

print(final_df[['name', 'price_steam', 'price_steam_usd', 'price_gog', 'price_usd']].head(10))
print(final_df['price_usd'].describe())

                         name  price_steam  price_steam_usd  price_gog  \
0             007 First Light        69.99        80.873445        NaN   
1           100% Orange Juice         6.99         8.076945        NaN   
2                1348 Ex Voto        22.49        25.987195        NaN   
3     14 Minesweeper Variants         6.99         8.076945        NaN   
4                         171        19.99        23.098445        NaN   
5                     1根香腸走江湖         0.00         0.000000        NaN   
6        20 Minutes Till Dawn         4.99         5.765945        NaN   
7           200% Mixed Juice!         6.99         8.076945        NaN   
8                        20XX         2.24         2.588320        NaN   
9  20XX - Draco Character DLC         1.49         1.721695        NaN   

   price_usd  
0  80.873445  
1   8.076945  
2  25.987195  
3   8.076945  
4  23.098445  
5   0.000000  
6   5.765945  
7   8.076945  
8   2.588320  
9   1.721695  
count    11168.00000

In [35]:
final_df.to_csv('FULL_dataset_full.csv', index=False)

eda_df = final_df[[
    'app_id', 'name', 'release_date', 'release_year',
    'price_usd', 'price_steam', 'price_gog',
    'steam_positive_pct', 'steam_total_reviews',
    'steam_review_label', 'rawg_rating', 'metacritic',
    'avg_playtime_hours', 'platforms_count', 'achievements_count',
    'esrb_rating', 'gog_rating', 'gog_reviews_count',
    'description', 'description_length', 'genres', 'tags', 'developer',
    'is_free', 'has_description', 'has_rawg', 'has_gog',
    'popularity_score_v2'
]].copy()

eda_df.to_csv('EDA_dataset.csv', index=False)